# Manual review: does attention entropy track question difficulty?

Samples questions from 3 sources with different, independent notions of difficulty coverage:
- **RACE++** (middle/high/race-c) — difficulty from which exam subset
- **OneStopQA** — difficulty from `level` (0/1/2 = elementary/intermediate/advanced), open-ended questions (not cloze)
- **SQuAD** — no difficulty tiers, used as a known-mostly-extractive baseline (should skew low entropy if the hypothesis holds)

For each sampled question: extract sentence-total attention distribution, plot it, show the entropy value, and manually judge whether it matches intuition. See `question_generation/docs/difficulty_steering_mechanisms.md` for the plan this feeds into.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

import random
import textwrap
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

from question_difficulty.methods.feature_based.difficulty_signals import AttentionDispersionSignal

random.seed(0)  # reproducible sampling across notebook re-runs

## Load a small balanced sample from each source

In [ ]:
N_PER_GROUP = 5  # keep small -- this is manual review, not bulk extraction

def sample_race(subset, n):
    ds = load_dataset("ehovy/race", subset, split="train")
    letter_to_idx = {"A": 0, "B": 1, "C": 2, "D": 3}
    items = []
    idxs = random.sample(range(len(ds)), min(n * 20, len(ds)))
    for i in idxs:
        rec = ds[i]
        if letter_to_idx.get(rec["answer"]) is None:
            continue
        items.append({"source": f"RACE-{subset}", "passage": rec["article"], "question": rec["question"]})
        if len(items) >= n:
            break
    return items

def sample_race_c(n):
    ds = load_dataset("tasksource/race-c", split="train")
    items = []
    idxs = random.sample(range(len(ds)), min(n * 20, len(ds)))
    for i in idxs:
        rec = ds[i]
        if rec["label"] is None:
            continue
        items.append({"source": "RACE-C", "passage": rec["article"], "question": rec["question"]})
        if len(items) >= n:
            break
    return items

def sample_onestopqa(level, n):
    ds = load_dataset("malmaud/onestop_qa", split="train")
    level_recs = [r for r in ds if r["level"] == level]
    picked = random.sample(level_recs, min(n, len(level_recs)))
    level_name = {0: "elementary", 1: "intermediate", 2: "advanced"}[level]
    return [{"source": f"OneStopQA-{level_name}", "passage": r["paragraph"], "question": r["question"]} for r in picked]

def sample_squad(n):
    ds = load_dataset("rajpurkar/squad", split="train")
    idxs = random.sample(range(len(ds)), n)
    return [{"source": "SQuAD", "passage": ds[i]["context"], "question": ds[i]["question"]} for i in idxs]

samples = []
samples += sample_race("middle", N_PER_GROUP)
samples += sample_race("high", N_PER_GROUP)
samples += sample_race_c(N_PER_GROUP)
samples += sample_onestopqa(0, N_PER_GROUP)
samples += sample_onestopqa(1, N_PER_GROUP)
samples += sample_onestopqa(2, N_PER_GROUP)
samples += sample_squad(N_PER_GROUP)

print(f"Total samples: {len(samples)}")
for s in samples[:3]:
    print(f"  [{s['source']}] {s['question'][:70]}")

## QA model candidates

All verified to actually load (never trust a model ID without checking --
`mrm8488/distilroberta-base-finetuned-squad`, previously used in this
project, currently 404s on HF Hub; excluded here, needs a separate fix in
`question_answering/docs/qa_model_battery.md`).

`vasudevgupta/bigbird-roberta-natural-questions` and
`google/bigbird-base-trivia-itc` (Natural Questions / TriviaQA -- genuinely
non-SQuAD) were also considered but excluded: neither ships `safetensors`
weights, and the installed `transformers` now refuses to load legacy
`pytorch_model.bin` checkpoints without `torch>=2.6` (CVE-2025-32434). Not
worth a project-wide torch upgrade just for this comparison.

`consciousAI/question-answering-roberta-base-s-v2` is the one non-SQuAD-named
model that actually loads and runs cleanly (standard RoBERTa, same dense
attention as the others) -- included below, though its exact training data
mix isn't independently confirmable from here (no model-card browsing access
in this environment), only that it isn't named/marketed as SQuAD-only like
the others.

Loads all 4 up front so any cell below can pick which one to use for
comparison, without re-downloading mid-session.

In [ ]:
QA_MODEL_CANDIDATES = {
    "roberta-base-squad2": "deepset/roberta-base-squad2",       # standard baseline
    "deberta-v3-base-squad2": "deepset/deberta-v3-base-squad2", # stronger
    "distilbert-squad": "distilbert-base-cased-distilled-squad", # weaker/smaller, for contrast
    "roberta-nonsquad": "consciousAI/question-answering-roberta-base-s-v2", # not SQuAD-named
}

signals = {name: AttentionDispersionSignal(qa_model_name=model_id)
           for name, model_id in QA_MODEL_CANDIDATES.items()}

QA_MODEL = "roberta-base-squad2"  # <-- change this to switch which model the extraction cell below uses

## Extract sentence-total attention + entropy for every sample (single layer, from `QA_MODEL`)

In [ ]:
LAYER = 11  # <-- change this to inspect a different layer

signal = signals[QA_MODEL]
results = []
for s in samples:
    detail = signal.get_sentence_distribution(s["passage"], s["question"], layer=LAYER)
    results.append({**s, **detail})

results.sort(key=lambda r: r["entropy"] if r["entropy"] is not None else -1)
print(f"Sorted low -> high entropy ({QA_MODEL}, layer {LAYER}):")
for r in results:
    print(f"  entropy={r['entropy']:.3f}  [{r['source']}]  {r['question'][:70]}")

## Single-layer view: pick one example, one layer

Bar chart = attention mass per sentence (sentence 0 = start of passage). Re-run with a different `IDX` to page through the sorted list above.

In [ ]:
IDX = 0  # <-- change this to page through `results` (sorted low->high entropy)

r = results[IDX]
print(f"[{r['source']}]  entropy={r['entropy']:.3f}  (model={QA_MODEL}, layer={r['layer']})")
print(f"Q: {r['question']}")
print()
for i, sent in enumerate(r["sentences"]):
    print(f"  S{i} ({r['distribution'][i]:.2f}): {textwrap.shorten(sent, 100)}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(r["distribution"])), r["distribution"])
ax.set_xlabel("sentence index")
ax.set_ylabel("attention mass")
ax.set_title(f"{r['source']} — entropy={r['entropy']:.3f}")
plt.show()

## All-layers comparison view

For the same `IDX` selected above: heatmap of attention mass (rows = layer 0-11, cols = sentence index) plus an entropy-per-layer line plot, so you can see at a glance which layers concentrate vs. spread for this specific question.

In [ ]:
all_layers = signal.get_all_layers_distribution(r["passage"], r["question"])
dist_matrix = np.array(all_layers["distributions"])  # (12, n_sentences)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [2, 1]})

im = ax1.imshow(dist_matrix, aspect="auto", cmap="viridis")
ax1.set_xlabel("sentence index")
ax1.set_ylabel("layer")
ax1.set_title(f"{r['source']} — attention mass per layer x sentence ({QA_MODEL})")
fig.colorbar(im, ax=ax1, label="attention mass")

ax2.plot(range(12), all_layers["entropies"], marker="o")
ax2.set_xlabel("layer")
ax2.set_ylabel("entropy")
ax2.set_title("entropy per layer")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Notes / manual verdicts

Record judgments here as you page through -- e.g. `IDX=0: matches (simple factual lookup)`, `IDX=12: surprising, seems easy but scored high`.

In [ ]:
# verdicts = {}
# verdicts[0] = "matches"
